# Windowed Analysis Tutorial

This tutorial provides a deep dive into Neurodent's windowed analysis capabilities for extracting features from continuous EEG data.

## Overview

Windowed Analysis Results (WAR) is the core feature extraction system in Neurodent. It:

1. Divides continuous EEG data into time windows
2. Computes features for each window
3. Aggregates results across time and channels
4. Provides filtering and quality control methods

This approach is efficient for long recordings and enables parallel processing.

In [1]:
import sys
from pathlib import Path
import logging
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from neurodent import core, visualization, constants

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()

/mnt/isilon/marsh_single_unit/YY_PyEEG/neurodent_Yong/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Feature Categories

Neurodent extracts four main categories of features:

### Linear Features (per channel)
Single-value metrics for each channel in each time window:

In [2]:
# Available linear features
print("Linear features:")
for feature in constants.LINEAR_FEATURES:
    print(f"  - {feature}")

# Examples:
# - rms: Root mean square amplitude
# - logrms: Log of RMS amplitude
# - ampvar: Amplitude variance
# - psdtot: Total power spectral density
# - psdslope: Slope of PSD on log-log scale

Linear features:
  - rms
  - ampvar
  - psdtotal
  - psdslope
  - nspike
  - logrms
  - logampvar
  - logpsdtotal
  - lognspike


### Band Features (per frequency band)
Features computed for each frequency band (delta, theta, alpha, beta, gamma):

In [3]:
# Available band features
print("\nBand features:")
for feature in constants.BAND_FEATURES:
    print(f"  - {feature}")

# Frequency bands
print("\nFrequency bands:")
for band, (lo, hi) in constants.FREQ_BANDS.items():
    print(f"  {band.capitalize()}: {lo}-{hi} Hz")


Band features:
  - psdband
  - psdfrac
  - logpsdband
  - logpsdfrac

Frequency bands:
  Delta: 0.1-4 Hz
  Theta: 4-8 Hz
  Alpha: 8-13 Hz
  Beta: 13-25 Hz
  Gamma: 25-40 Hz


### Matrix Features (connectivity)
Features measuring relationships between channels:

In [4]:
# Available matrix features
print("\nMatrix features:")
for feature in constants.MATRIX_FEATURES:
    print(f"  - {feature}")

# Examples:
# - cohere: Spectral coherence between channel pairs
# - pcorr: Pearson correlation between channels


Matrix features:
  - cohere
  - zcohere
  - imcoh
  - zimcoh
  - pcorr
  - zpcorr


## 2. Computing Windowed Analysis

### Basic Usage

In [ ]:
# Using included test data
data_path = Path("../../.tests/integration/data/A10/A10_recording.edf")
animal_id = "A10"

# Create LongRecordingOrganizer with SpikeInterface mode
# mode options: 'si' (SpikeInterface), 'mne', or None
lro = core.LongRecordingOrganizer(
    item=data_path,
    mode="si",
    extract_func="read_edf",
    manual_datetimes=datetime(2023, 12, 13),
)

# Create AnimalOrganizer using pattern-based discovery
ao = visualization.AnimalOrganizer(
    pattern="../../.tests/integration/data/{animal}/*.edf",
    animal_id=animal_id,
    assume_from_number=True,
    lro_kwargs={
        "mode": "si",
        "extract_func": "read_edf",
        "manual_datetimes": datetime(2023, 12, 13),
    },
)

# Compute all features
war_all = ao.compute_windowed_analysis(
    features=['all'],
    exclude=['nspike', 'lognspike'],
    multiprocess_mode='serial'
)

INFO:root:CSV metadata timestamps: 1 of 1 files have timestamps
INFO:root:Converting 1 column-major binary files to row-major format
INFO:root:Overwrite flag not set - only generating missing row-major files
INFO:root:Recording already at target sampling rate (1000 Hz), no resampling needed
INFO:root:LongRecording created: LongRecording: 1 files, 10 channels, 1000.0 Hz, 120.4s duration, channels: [Intan Input (1)/PortC C-009, Intan Input (1)/PortC C-010, Intan Input (1)/PortC C-012, Intan Input (1)/PortC C-014, Intan Input (1)/PortC C-015, Intan Input (1)/PortC C-016, Intan Input (1)/PortC C-017, Intan Input (1)/PortC C-019, Intan Input (1)/PortC C-021, Intan Input (1)/PortC C-022], float32 precision, µV units, timestamps: 1/1 files have timestamps
INFO:root:Finalizing file timestamps
INFO:root:Using CSV metadata timestamps
INFO:root:bin_folder_pattern: ../../notebooks/tests/test-data/*A10 KO 12_13_2023*
INFO:root:self._bin_folders: ['../../notebooks/tests/test-data/A10 KO 12_13_2023']

### Access information in WindowedAnalysis object

You can access the summary of WindowedAnalysis object using the `get_info` method, and access the computed features in the WindowedAnalysis object using the `get_result` method.

In [ ]:
war_info = war_selective.get_info()
print(war_info)

war_result = war_selective.get_result(features=["all"])
print(war_result)

### Selective Feature Computation

For faster processing, compute only needed features. In this example, we will get 'rms' and 'psdband':

In [ ]:
war_selective.get_info()

In [ ]:
# Compute specific features
# war_selective = ao.compute_windowed_analysis(
#     features=['rms', 'logrms', 'psdband', 'cohere'],
#     multiprocess_mode='serial'
# )

results = war_selective.get_result(features=['all'])

# print(f"Computed features: {war_selective.}")

### Parallel Processing

For large datasets, it is recommended to use parallel processing by setting 'multiprocess_mode' argument.

You can either use:
1) All local CPU cores ('serial'), or
2) Dask for distributed computing ('dask').

In [21]:
# Option 1: Serial (uses all CPU cores)
war_serial = ao.compute_windowed_analysis(
    features=['rms', 'psdband'],
    multiprocess_mode='serial'
)

print(war_serial.get_info())

# Option 2: Dask (for distributed computing)
# Requires Dask cluster setup
# war_dask = ao.compute_windowed_analysis(
#     features=['rms', 'psdband'],
#     multiprocess_mode='dask'
# )

INFO:root:Computing windowed analysis for ../../notebooks/tests/test-data/A10 KO 12_13_2023
Processing rows:  45%|████▌     | 14/31 [00:00<00:00, 134.64it/s]/mnt/isilon/marsh_single_unit/YY_PyEEG/neurodent_Yong/.venv/lib/python3.10/site-packages/scipy/signal/_spectral_py.py:790: UserWarning: nperseg = 1000 is greater than input length  = 360, using nperseg = 360
  freqs, _, Pxy = _spectral_helper(x, y, fs, window, nperseg, noverlap,
Processing rows: 100%|██████████| 31/31 [00:00<00:00, 157.14it/s]
INFO:root:Total LOF scores collected: 0 animal days
/mnt/isilon/marsh_single_unit/YY_PyEEG/neurodent_Yong/src/neurodent/visualization/results.py:1436: UserWarning: WARNING: 1 animalday(s) are missing LOF scores: ['A10 KO 12_13_2023 KO Jan-01-2000']. Expected 1 but got 0. These sessions will be auto-populated with empty placeholders and excluded from LOF-based analysis.
  warnings.warn(warning_msg)
INFO:root:Added missing animalday to bad_channels_dict: A10 KO 12_13_2023 KO Jan-01-2000
/mnt/is

feature names: rms, psdband
animaldays: A10 KO 12_13_2023 KO Jan-01-2000
animal_id: A10 KO 12_13_2023
genotype: KO
channel_names: Intan Input (1)/PortC C-009, Intan Input (1)/PortC C-010, Intan Input (1)/PortC C-012, Intan Input (1)/PortC C-014, Intan Input (1)/PortC C-015, Intan Input (1)/PortC C-016, Intan Input (1)/PortC C-017, Intan Input (1)/PortC C-019, Intan Input (1)/PortC C-021, Intan Input (1)/PortC C-022


## 3. Data Quality and Filtering

### Method Chaining (Recommended)

Apply multiple filters in sequence:

In [22]:
war_filtered = (
    war_all
    .filter_logrms_range(z_range=3)           # Remove outliers (±3 SD)
    .filter_high_rms(max_rms=500)             # Remove high amplitude artifacts
    .filter_low_rms(min_rms=10)               # Remove low amplitude periods
    # .filter_high_beta(max_beta_prop=0.4)      # Remove high beta activity
    .filter_reject_channels_by_session()      # Reject bad channels
    .filter_morphological_smoothing(smoothing_seconds=4.0)  # Smooth filter mask (fill short gaps, remove brief artifacts)
)

print("Filtering completed!")

INFO:root:Filtering rms
INFO:root:Filtering ampvar
INFO:root:Filtering psdtotal
INFO:root:Filtering psdslope
INFO:root:Skipping nspike because it is not in result
INFO:root:Filtering logrms
INFO:root:Filtering logampvar
INFO:root:Filtering logpsdtotal
INFO:root:Skipping lognspike because it is not in result
INFO:root:Filtering psdband
INFO:root:Filtering psdfrac
INFO:root:Filtering logpsdband
INFO:root:Filtering logpsdfrac
INFO:root:Filtering cohere
INFO:root:Filtering zcohere
INFO:root:Filtering imcoh
INFO:root:Filtering zimcoh
INFO:root:Filtering pcorr
INFO:root:Filtering zpcorr
INFO:root:Filtering psd
INFO:root:set([x[0].shape for x in result[feat].tolist()]) = [(501,)]
INFO:root:set([x[1].shape for x in result[feat].tolist()]) = [(501, 10)]
/mnt/isilon/marsh_single_unit/YY_PyEEG/neurodent_Yong/src/neurodent/visualization/results.py:2179: UserWarning: One or more channels do not match name aliases. Assuming alias from number in channel name.
  core.parse_chname_to_abbrev(x, assume_f

Filtering completed!


### Configuration-Driven Filtering

Alternative approach using configuration dictionary:

In [23]:
filter_config = {
    'logrms_range': {'z_range': 3},
    'high_rms': {'max_rms': 500},
    'low_rms': {'min_rms': 10},
    # 'high_beta': {'max_beta_prop': 0.4},
    'reject_channels_by_session': {},
    'morphological_smoothing': {'smoothing_seconds': 4.0}
}

war_filtered_config = war_all.apply_filters(
    filter_config,
    min_valid_channels=3
)

INFO:root:logrms_range: filtered 2/310
INFO:root:high_rms: filtered 0/310
INFO:root:low_rms: filtered 104/310
INFO:root:high_beta: filtered 10/310
INFO:root:reject_channels_by_session: filtered 0/310
INFO:root:Applied morphological smoothing: 8.0s
INFO:root:Filtering rms
INFO:root:Filtering ampvar
INFO:root:Filtering psdtotal
INFO:root:Filtering psdslope
INFO:root:Skipping nspike because it is not in result
INFO:root:Filtering logrms
INFO:root:Filtering logampvar
INFO:root:Filtering logpsdtotal
INFO:root:Skipping lognspike because it is not in result
INFO:root:Filtering psdband
INFO:root:Filtering psdfrac
INFO:root:Filtering logpsdband
INFO:root:Filtering logpsdfrac
INFO:root:Filtering cohere
INFO:root:Filtering zcohere
INFO:root:Filtering imcoh
INFO:root:Filtering zimcoh
INFO:root:Filtering pcorr
INFO:root:Filtering zpcorr
INFO:root:Filtering psd
INFO:root:set([x[0].shape for x in result[feat].tolist()]) = [(501,)]
INFO:root:set([x[1].shape for x in result[feat].tolist()]) = [(501, 10

### Available Filters

- `filter_logrms_range(z_range)`: Remove outliers based on log RMS
- `filter_high_rms(max_rms)`: Remove high amplitude artifacts
- `filter_low_rms(min_rms)`: Remove low amplitude periods
- `filter_high_beta(max_beta_prop)`: Remove high beta activity (muscle artifacts)
- `filter_reject_channels_by_session()`: Identify and reject bad channels
- `morphological_smoothing(smoothing_seconds)`: Smooth data morphologically

## 4. Data Aggregation

Average across time windows, producing a single row per group (e.g., per recording session and light/dark phase). Channel information is preserved.

In [24]:
# Aggregate time windows
war_filtered.aggregate_time_windows()

/mnt/isilon/marsh_single_unit/YY_PyEEG/neurodent_Yong/src/neurodent/visualization/results.py:4065: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  aggregated_df = result_grouped.apply(
INFO:root:Setting suppress_short_interval_error to True
/mnt/isilon/marsh_single_unit/YY_PyEEG/neurodent_Yong/src/neurodent/visualization/results.py:2179: UserWarning: One or more channels do not match name aliases. Assuming alias from number in channel name.
  core.parse_chname_to_abbrev(x, assume_from_number=self.assume_from_number)


## 5. Channel Management

### Reorder and Pad Channels

Ensure consistent channel ordering across animals:

In [ ]:
# Define standard channel order
standard_channels = [
    "LMot", "RMot",  # Motor cortex
    "LBar", "RBar",  # Barrel cortex
    "LAud", "RAud",  # Auditory cortex
    "LVis", "RVis",  # Visual cortex
    "LHip", "RHip"   # Hippocampus
]

war_filtered.reorder_and_pad_channels(
    standard_channels,
    use_abbrevs=True  # Use abbreviated channel names
)

print(f"Channels: {war_filtered.channel_names}")

Channels: ['LMot', 'RMot', 'LBar', 'RBar', 'LAud', 'RAud', 'LVis', 'RVis', 'LHip', 'RHip']


## 6. Accessing Computed Features

WAR objects store features in a pandas DataFrame. Use `get_result()` to retrieve features with full channel information, or `get_channel_averaged_result()` to average across channels (or channel pairs for connectivity features), producing scalar values per time window.

In [ ]:
print(war_filtered._feature_columns)


['rms', 'ampvar', 'psdtotal', 'psdslope', 'logrms', 'logampvar', 'logpsdtotal', 'psdband', 'psdfrac', 'logpsdband', 'logpsdfrac', 'cohere', 'zcohere', 'imcoh', 'zimcoh', 'pcorr', 'zpcorr', 'psd']


In [63]:
rms_data = war_filtered.result['rms'][0]
rms_data
#print(f"RMS shape: {rms_data.shape}")
# print(f"RMS dims: {rms_data.ndim}")
# print(f"RMS coords: {list(rms_data.coords)}")

[np.float64(56.674784088134764),
 np.float64(91.52802454630533),
 np.float64(87.12390696207682),
 np.float64(75.27189509073894),
 np.float64(nan),
 np.float64(78.75343246459961),
 np.float64(92.8258455912272),
 np.float64(143.3729685465495),
 np.float64(nan),
 np.float64(54.32293542226156)]

In [65]:
war_filtered.result['rms'][0]

[np.float64(56.674784088134764),
 np.float64(91.52802454630533),
 np.float64(87.12390696207682),
 np.float64(75.27189509073894),
 np.float64(nan),
 np.float64(78.75343246459961),
 np.float64(92.8258455912272),
 np.float64(143.3729685465495),
 np.float64(nan),
 np.float64(54.32293542226156)]

In [ ]:
# Access RMS data
rms_data = war_filtered.rms
print(f"RMS shape: {rms_data.shape}")
print(f"RMS dims: {rms_data.dims}")
print(f"RMS coords: {list(rms_data.coords)}")

# Access band power data
psdband_data = war_filtered.psdband
print(f"\nPSD Band shape: {psdband_data.shape}")
print(f"PSD Band dims: {psdband_data.dims}")
print(f"Bands: {list(psdband_data.coords['band'].values)}")

# Access coherence data (matrix feature)
cohere_data = war_filtered.cohere
print(f"\nCoherence shape: {cohere_data.shape}")
print(f"Coherence dims: {cohere_data.dims}")

## 7. Metadata and Grouping Variables

WAR objects contain metadata for grouping and analysis:

In [ ]:
# Access metadata
print(f"Animal ID: {war_filtered.animal_id}")
print(f"Genotype: {war_filtered.genotype}")
print(f"Animal days: {war_filtered.animaldays}")
print(f"Channel names: {war_filtered.channel_names}")

## 8. Circadian Analysis (ZeitgeberAnalysisResult)

Once your data is filtered and metadata (like genotype and timestamps) is verified, you can analyze circadian rhythms. 

The `ZeitgeberAnalysisResult` wrapper uses this metadata to:
1.  **Shift Timestamps**: Converts absolute timestamps to Zeitgeber Time (ZT), where ZT0 is "Lights On".
2.  **Define Baseline**: Subtracts a baseline period (e.g., the first 12 hours of the light phase) to normalization data.
3.  **Prepare for Visualization**: Duplicates data for 48-hour "double-plotted" actograms.

For plotting these results, see the **[Visualization Tutorial](visualization.ipynb)**.

In [ ]:
from neurodent import core

# Wrap the result for circadian analysis
zar = core.ZeitgeberAnalysisResult(
    war_filtered,
    baseline_hours=12,
    zeitgeber_shift_hours=6,
    shift_for_48h=True
)

# Access the processed data
df_zar = zar.get_result(features=['rms'])

timestamps = war_filtered.result['timestamp']
print(f"Original Time Range: {timestamps.min()} to {timestamps.max()}")
print(f"ZT Coordinate Range: {df_zar.total_minutes.min()} to {df_zar.total_minutes.max()} min")
print(f"Baseline-Corrected Columns: {[c for c in df_zar.columns if '_nobase' in c][:3]}")

## 9. Saving and Loading

Save WAR objects for later analysis:

In [ ]:
import tempfile

# Use a temporary directory for demonstration purposes
with tempfile.TemporaryDirectory() as tmpdir:
    output_path = Path(tmpdir) / animal_id
    output_path.mkdir(parents=True, exist_ok=True)

    # Save WAR
    war_filtered.save_pickle_and_json(output_path)
    print(f"Saved to {output_path}")

    # Load WAR
    war_loaded = visualization.WindowAnalysisResult.load_pickle_and_json(output_path)
    print(f"Loaded from {output_path}")


## 10. Best Practices

### Feature Selection
- Start with basic features (rms, psdband) before computing expensive ones (cohere, psd)
- Exclude spike features if you don't have spike data
- Use selective feature computation for faster iteration

### Filtering
- Always inspect data before and after filtering
- Use conservative thresholds initially, then adjust
- Consider biological significance (e.g., high beta may indicate muscle artifacts)

### Processing
- Use serial mode for debugging
- Use multiprocess for local analysis of large datasets
- Use Dask for cluster computing

### Quality Control
- Check channel consistency across animals
- Verify metadata (genotype, day, etc.)
- Save intermediate results frequently

## Summary

This tutorial covered:

1. Feature categories and types
2. Computing windowed analysis with different options
3. Data quality control and filtering
4. Channel management and standardization
5. Accessing computed features
6. Metadata and grouping variables
7. Saving and loading results
8. Best practices

## Next Steps

- **[Visualization Tutorial](visualization.ipynb)**: Plot and analyze WAR results
- **[Spike Analysis Tutorial](spike_analysis.ipynb)**: Integrate spike-sorted data